# Proyecto Sant Andreu — Scraping BeSoccer
## Segunda RFEF Grupo 3 · Temporada 2025-26

---

**Autor:** Manuel Pérez Ruda  
**Fecha:** Febrero 2026  
**Máster:** Sports Data Campus — Departamento de Extensión Profesional

---

### Objetivo del Proyecto

Construir un **pipeline ETL completo** para extraer, transformar y cargar datos de fútbol
desde BeSoccer, generando un modelo dimensional (Star Schema) listo para análisis en Power BI.

### Datos Extraídos

- **Equipos**: 18 equipos del Grupo 3
- **Partidos**: Todas las jornadas de la temporada
- **Alineaciones**: Titulares y suplentes con dorsal y posición
- **Goles**: Goleadores, asistentes, minuto y tipo de gol
- **Sustituciones**: Jugador que entra, que sale y minuto

### Uso habitual — Actualización semanal (cada lunes)

Ejecutar las celdas en orden:
1. **Configuración inicial** (sección 3) — siempre
2. **Pipeline incremental** (sección 7) — solo procesa los partidos nuevos
3. **Revisión de resultados** (sección 9) — verificar que todo es correcto
4. **Exportación a Power BI** (sección 11) — actualizar el dashboard

---
## 1. Arquitectura del Proyecto

```
Proyecto Sant Andreu/
├── src/
│   ├── config.py             # Configuración centralizada (rutas, URLs, HTTP)
│   ├── utils/                # HTTP client, logging, parquet utils
│   ├── scrapers/             # Descarga HTML de BeSoccer
│   ├── parsers/              # HTML → objetos Python
│   ├── transform/            # Objetos → DataFrames Parquet
│   └── pipeline/             # Orquestador del pipeline
├── html/                     # Caché HTML permanente (no borrar)
│   ├── lineups/              #   <match_id>.html — alineaciones
│   └── events/               #   <match_id>.html — goles y sustituciones
├── data_processed/           # Tablas intermedias en Parquet
│   └── meta/                 #   processed_matches.parquet
├── outputs_powerbi/          # Tablas para Power BI (nombres en mayúsculas)
├── logs/                     # Logs de cada ejecución
└── notebooks/                # Este notebook
```

---
## 2. Modelo de Datos (Star Schema)

```
                   +--------------+
                   |  DIM_EQUIPOS |
                   |  (18 equipos)|
                   +------+-------+
                          |
         +-----------+----+----+-----------+
         |           |         |           |
         v           v         v           v
    +----------+ +--------+ +--------+ +--------+
    |FACT_GOALS| |DIM_PART| |FACT_SUB| |FACT_APP|
    |  (goles) | |(part.) | |(subs)  | |(alin.) |
    +----------+ +--------+ +--------+ +---+----+
                                            |
                                            v
                                    +---------------+
                                    | DIM_JUGADORES |
                                    | (~600 jugad.) |
                                    +---------------+
```

| Tabla | Clave primaria | Claves foráneas |
|-------|---------------|-----------------|
| `DIM_EQUIPOS` | `team_id` | — |
| `DIM_PARTIDOS` | `match_id` | `home_team_id`, `away_team_id` |
| `DIM_JUGADORES` | `player_sk` | `team_id` |
| `FACT_GOALS` | `match_id + goal_index` | `match_id`, `team_id`, `scorer_player_sk` |
| `FACT_SUBSTITUTIONS` | `match_id + minute + player_in_sk` | `match_id`, `team_id` |
| `FACT_APPEARANCES` | `match_id + player_sk` | `match_id`, `team_id`, `player_sk` |

---
## 3. Configuración Inicial

Añadimos `src/` al path de Python e importamos la clase `Config`, que centraliza
todas las rutas, URLs y parámetros del proyecto. No hay valores hardcodeados
fuera de este módulo.

In [ ]:
import sys
import pandas as pd
from pathlib import Path

# Ruta raíz del proyecto
PROJECT_DIR = Path(r"C:\Users\manue\OneDrive\Sports Data Campus\Máster Python\Proyecto Sant Andreu")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from config import get_config
config = get_config()

print(f"Proyecto : {config.COMPETITION_NAME} {config.GROUP}")
print(f"Temporada: {config.SEASON}")
print(f"Base dir : {config.BASE_DIR}")
print(f"Caché lineups: {config.LINEUPS_HTML_DIR}")
print(f"Caché events : {config.EVENTS_HTML_DIR}")

# Garantizar que los directorios de caché existen
config.LINEUPS_HTML_DIR.mkdir(parents=True, exist_ok=True)
config.EVENTS_HTML_DIR.mkdir(parents=True, exist_ok=True)
print("\nDirectorios de caché listos.")

---
## 4. Sistema de Logging

El pipeline registra cada operación en `logs/pipeline_YYYYMMDD.log`.
Al activar el logging aquí, los mensajes aparecen también en la salida del notebook,
lo que permite seguir el progreso en tiempo real durante el scraping.

In [ ]:
import logging
from utils.logging_config import setup_logging

logger = setup_logging(config.LOGS_DIR, level=logging.INFO)
print(f"Log activo → {config.LOGS_DIR}")

---
## 5. Extracción de Equipos (DIM_EQUIPOS)

Scrapeamos la página de clasificación del Grupo 3 en BeSoccer.
De cada equipo extraemos:

- `team_id` — del número en la URL del escudo (p.ej. `.../equipos/1981.png` → `1981`)
- `nombre_equipo` — nombre oficial en BeSoccer (es el que hay que usar en el videoanálisis)
- `slug` — parte de la URL del equipo, usado para construir las URLs de partidos
- `posicion_actual` — posición en la clasificación en el momento del scraping

Si `team_id = None` para algún equipo, el log muestra una advertencia — puede indicar
un cambio en el formato de URLs del CDN de BeSoccer.

In [ ]:
from transform.dim_equipos import update_dim_equipos

df_equipos = update_dim_equipos()

print(f"Total equipos: {len(df_equipos)}")
print()
df_equipos[["posicion_actual", "nombre_equipo", "team_id", "slug"]].sort_values("posicion_actual")

---
## 6. Extracción de Partidos (DIM_PARTIDOS)

Para cada uno de los 18 equipos scrapeamos su página de partidos en BeSoccer,
filtrando los de la temporada 2025-26 y de la Segunda Federación.

Como cada partido aparece en la página de los dos equipos que se enfrentan,
el sistema deduplica por `match_id` — solo se guarda un registro por partido.

Cada partido tiene:
- `match_id` — identificador único de BeSoccer
- `home_team_id` / `away_team_id` — IDs de los equipos
- `score_home` / `score_away` — resultado final
- `status` — `Finalizado`, `Programado` o `Aplazado`
- `jornada` — número de jornada
- `match_url` — URL del partido (usada después para scrapear alineaciones y eventos)

In [ ]:
from transform.dim_partidos import update_dim_partidos

df_partidos = update_dim_partidos()

print(f"Total partidos: {len(df_partidos)}")
print(f"  Finalizados: {(df_partidos['status'] == 'Finalizado').sum()}")
print(f"  Programados: {(df_partidos['status'] == 'Programado').sum()}")
print(f"  Aplazados:   {(df_partidos['status'] == 'Aplazado').sum()}")

print("\nÚltimos 5 partidos finalizados:")
finalizados = df_partidos[df_partidos["status"] == "Finalizado"].sort_values("fecha", ascending=False)
finalizados[["jornada", "home_team_name", "score_home", "score_away", "away_team_name", "fecha"]].head()

---
## 7. Pipeline de Scraping — Alineaciones y Eventos

Este es el paso principal. Para cada partido finalizado que aún **no ha sido procesado**,
el pipeline realiza lo siguiente en orden:

1. Descarga la página `/alineaciones` del partido y extrae titulares y suplentes
2. Descarga la página principal del partido y extrae goles y sustituciones
3. Guarda ambos HTMLs en `html/lineups/` y `html/events/` con nombre `<match_id>.html`
4. Enriquece el `minute_in` de cada suplente cruzando con los datos de sustituciones
5. Construye o actualiza `FACT_APPEARANCES`, `FACT_GOALS` y `FACT_SUBSTITUTIONS`
6. Reconstruye `DIM_JUGADORES` a partir de las apariciones
7. Registra el partido como procesado en `meta/processed_matches.parquet`

**En las siguientes ejecuciones semanales, `run_incremental()` solo procesa
los partidos nuevos** — los ya procesados se saltan automáticamente.
Los HTMLs cacheados evitan re-descargar páginas ya visitadas.

In [ ]:
from pipeline.run_pipeline import run_incremental

# Ejecuta el pipeline incremental: solo procesa los partidos nuevos
stats = run_incremental()

print("\n" + "=" * 50)
print("RESUMEN DEL PIPELINE")
print("=" * 50)
print(f"  Partidos nuevos procesados: {stats['partidos_procesados']}")
print(f"  Goles totales en BBDD:      {stats['goles']}")
print(f"  Sustituciones totales:      {stats['sustituciones']}")
print(f"  Apariciones totales:        {stats['apariciones']}")
print(f"  Errores:                    {stats['errores']}")

---
## 8. Estado del Pipeline

Muestra cuántos partidos hay finalizados, cuántos ya han sido procesados
y si queda alguno pendiente.

In [ ]:
from pipeline.processed_matches import get_processed_match_ids
from transform.dim_partidos import get_finished_matches

processed_ids = get_processed_match_ids()
df_finished   = get_finished_matches()

n_proc    = len(processed_ids)
n_fin     = len(df_finished)
n_pending = n_fin - n_proc

print(f"Partidos finalizados:  {n_fin}")
print(f"Partidos procesados:   {n_proc}")
print(f"Partidos pendientes:   {n_pending}")

if n_pending > 0:
    pendientes = df_finished[~df_finished["match_id"].isin(processed_ids)]
    print("\nPartidos pendientes de procesar:")
    print(pendientes[["jornada", "home_team_name", "score_home",
                       "score_away", "away_team_name"]].to_string(index=False))
else:
    print("\nTodo al día. No hay partidos pendientes.")

---
## 9. Revisión de Resultados

Verificamos el contenido de cada tabla para comprobar que el pipeline ha funcionado
correctamente y los datos son coherentes.

### 9.1 Goles (FACT_GOALS)

Cada fila es un gol con su goleador, asistente opcional, minuto, tipo
(`normal`, `penalty`, `own_goal`) y equipo.

La columna `goal_index` es el ordinal del gol dentro del partido (0, 1, 2...).
Actúa como parte de la clave del upsert junto con `match_id`, lo que permite
distinguir dos goles del mismo jugador en el mismo minuto sin perder ninguno.

In [ ]:
df_goals = pd.read_parquet(config.FACT_GOALS_PATH)

print(f"Total goles: {len(df_goals)}")
print(f"  Normales:       {(df_goals['goal_type'] == 'normal').sum()}")
print(f"  Penaltis:       {(df_goals['goal_type'] == 'penalty').sum()}")
print(f"  En propia:      {(df_goals['goal_type'] == 'own_goal').sum()}")
print(f"  Con asistencia: {df_goals['assist_player_sk'].notna().sum()}")

print("\nTop 10 Goleadores:")
top = df_goals.groupby("scorer_name").size().sort_values(ascending=False).head(10)
for i, (name, n) in enumerate(top.items(), 1):
    print(f"  {i:2}. {name}: {n} goles")

---
### 9.2 Sustituciones (FACT_SUBSTITUTIONS)

Cada fila es una sustitución con el jugador que entra, el que sale, el minuto y el equipo.

Esta tabla cumple una doble función:
- Es la **fuente canónica del `minute_in`** para `FACT_APPEARANCES`: cuando el pipeline
  procesa un partido, cruza cada suplente que entró con su sustitución para asignarle
  el minuto de entrada. Así hay una única fuente de verdad para ese dato.
- En Power BI permite calcular los **minutos jugados** de cada jugador combinándola con
  `FACT_APPEARANCES`.

In [ ]:
df_subs = pd.read_parquet(config.FACT_SUBSTITUTIONS_PATH)

print(f"Total sustituciones: {len(df_subs)}")
print(f"Minuto promedio:     {df_subs['minute'].mean():.1f}'")

franjas = pd.cut(df_subs["minute"],
                 bins=[0, 45, 60, 75, 90, 120],
                 labels=["0-45", "46-60", "61-75", "76-90", "90+"])
print("\nSustituciones por franja:")
print(franjas.value_counts().sort_index().to_string())

---
### 9.3 Apariciones (FACT_APPEARANCES)

Una fila por jugador por partido. Campos clave:

- `is_starter` — `True` para titulares, `False` para suplentes
- `minute_in` — `0` para titulares; para suplentes que entraron, el minuto de entrada
  (obtenido de `FACT_SUBSTITUTIONS`); `NULL` para suplentes que no llegaron a entrar
- `shirt_number` — dorsal, usado para resolver la identidad del jugador en el videoanálisis

**Cálculo de minutos jugados en Power BI (medida DAX):**

```dax
Minutos Jugados =
VAR MinIn  = SELECTEDVALUE(Fact_Appearances[minute_in])
VAR MinOut = CALCULATE(
    SELECTEDVALUE(Fact_Substitutions[minute]),
    Fact_Substitutions[player_out_sk] = SELECTEDVALUE(Fact_Appearances[player_sk]),
    Fact_Substitutions[match_id]      = SELECTEDVALUE(Fact_Appearances[match_id])
)
RETURN IF(ISBLANK(MinOut), 90 - MinIn, MinOut - MinIn)
```

In [ ]:
df_app = pd.read_parquet(config.FACT_APPEARANCES_PATH)

sup_entran = ((~df_app["is_starter"]) & df_app["minute_in"].notna()).sum()
sup_banco  = ((~df_app["is_starter"]) & df_app["minute_in"].isna()).sum()

print(f"Total apariciones:  {len(df_app)}")
print(f"  Titulares:        {df_app['is_starter'].sum()}")
print(f"  Suplentes entran: {sup_entran}")
print(f"  Suplentes banco:  {sup_banco}")

print("\nJugadores con más titularidades:")
top = df_app[df_app["is_starter"]].groupby("player_name").size().sort_values(ascending=False).head(10)
for i, (name, n) in enumerate(top.items(), 1):
    print(f"  {i:2}. {name}: {n}")

---
### 9.4 Jugadores (DIM_JUGADORES)

Se construye automáticamente agrupando `FACT_APPEARANCES` por `player_sk`.

La clave `player_sk` funciona así:
- Si el jugador tiene perfil en BeSoccer, `player_sk = player_id` (entero positivo)
- Si no tiene perfil, `player_sk` es un hash MD5 negativo calculado sobre
  `nombre + equipo` normalizados (sin tildes, en mayúsculas)

La normalización evita que "Javi Martínez" y "Javi Martinez" generen dos
entradas distintas en la dimensión.

In [ ]:
df_jug = pd.read_parquet(config.DIM_JUGADORES_PATH)

con_id = df_jug["player_id"].notna().sum()
sin_id = df_jug["player_id"].isna().sum()

print(f"Total jugadores únicos: {len(df_jug)}")
print(f"  Con player_id (BeSoccer): {con_id:4}  ({con_id/len(df_jug)*100:.1f}%)")
print(f"  Sin player_id (hash):     {sin_id:4}  ({sin_id/len(df_jug)*100:.1f}%)")

print("\nJugadores por equipo:")
por_equipo = (df_jug.groupby("team_id").size()
              .reset_index(name="jugadores")
              .merge(df_equipos[["team_id", "nombre_equipo"]], on="team_id", how="left")
              .sort_values("jugadores", ascending=False))
print(por_equipo[["nombre_equipo", "jugadores"]].to_string(index=False))

---
## 10. Caché HTML

Cada HTML descargado se guarda en `html/lineups/` o `html/events/` con el
nombre `<match_id>.html`. Esto proporciona dos ventajas:

1. **Sin re-scraping**: las ejecuciones semanales reutilizan el HTML de los partidos
   ya procesados — no se vuelve a hacer ninguna petición HTTP para ellos.
2. **Auditoría y re-parseo**: si en el futuro se mejora el parser (por ejemplo,
   para extraer datos adicionales), se puede re-procesar todo el historial sin
   volver a la web.

La celda siguiente verifica que la caché es coherente con los partidos procesados.

In [ ]:
lineups_html = list(config.LINEUPS_HTML_DIR.glob("*.html"))
events_html  = list(config.EVENTS_HTML_DIR.glob("*.html"))
total_kb     = sum(f.stat().st_size for f in lineups_html + events_html) / 1024

print(f"HTMLs de alineaciones: {len(lineups_html)}")
print(f"HTMLs de eventos:      {len(events_html)}")
print(f"Tamaño total caché:    {total_kb:.0f} KB")

from pipeline.processed_matches import get_processed_match_ids
proc_ids   = get_processed_match_ids()
lineup_ids = {int(f.stem) for f in lineups_html}
event_ids  = {int(f.stem) for f in events_html}

sin_lineup = proc_ids - lineup_ids
sin_events = proc_ids - event_ids

if sin_lineup:
    print(f"\nAVISO: {len(sin_lineup)} partidos procesados sin HTML de alineaciones")
else:
    print("\nOK: caché de alineaciones completa")

if sin_events:
    print(f"AVISO: {len(sin_events)} partidos procesados sin HTML de eventos")
else:
    print("OK: caché de eventos completa")

---
## 11. Exportación a Power BI

Las tablas de `data_processed/` se copian a `outputs_powerbi/` con los nombres
en mayúsculas que Power BI Desktop espera. Tras ejecutar esta celda, recargar
el informe `.pbix` para ver los datos actualizados.

| Archivo Power BI | Descripción | Clave |
|---|---|---|
| `DIM_EQUIPOS.parquet` | 18 equipos del Grupo 3 | `team_id` |
| `DIM_PARTIDOS.parquet` | Todos los partidos | `match_id` |
| `DIM_JUGADORES.parquet` | Jugadores únicos | `player_sk` |
| `FACT_GOALS.parquet` | Goles | `match_id + goal_index` |
| `FACT_SUBSTITUTIONS.parquet` | Sustituciones | `match_id + minute + player_in_sk` |
| `FACT_APPEARANCES.parquet` | Apariciones | `match_id + player_sk` |

In [ ]:
from utils.parquet_utils import export_all_to_powerbi

exported = export_all_to_powerbi()

print(f"Archivos exportados a outputs_powerbi/: {len(exported)}")
print()
for p in exported:
    df_tmp = pd.read_parquet(p)
    print(f"  {p.name:<35} {p.stat().st_size/1024:5.1f} KB  ({len(df_tmp):,} registros)")

---
## 12. Resumen de Archivos Generados

In [ ]:
print("data_processed/")
print("-" * 58)
for f in sorted(config.DATA_PROCESSED_DIR.glob("*.parquet")):
    df_tmp = pd.read_parquet(f)
    print(f"  {f.name:<38} {f.stat().st_size/1024:5.1f} KB  ({len(df_tmp):,} reg.)")

meta_f = config.PROCESSED_MATCHES_PATH
if meta_f.exists():
    df_meta = pd.read_parquet(meta_f)
    print(f"  {'meta/processed_matches.parquet':<38} "
          f"{meta_f.stat().st_size/1024:5.1f} KB  ({len(df_meta):,} reg.)")

print()
print("html/ (caché)")
print("-" * 58)
n_l = len(list(config.LINEUPS_HTML_DIR.glob("*.html")))
n_e = len(list(config.EVENTS_HTML_DIR.glob("*.html")))
print(f"  html/lineups/  {n_l} archivos")
print(f"  html/events/   {n_e} archivos")

---
## 13. Estructura de Módulos

```python
# Configuración — rutas, URLs, parámetros HTTP
from config import get_config

# Scrapers — descarga HTML de BeSoccer
from scrapers.classification import scrape_classification   # clasificación → equipos
from scrapers.matches import scrape_all_matches             # partidos por equipo
from scrapers.lineups import scrape_or_load_lineups         # alineaciones (con caché)
from scrapers.events import scrape_or_load_events           # goles y subs (con caché)

# Parsers — HTML → objetos Python
from parsers.lineup_parser import parse_lineup_html         # → PlayerAppearance
from parsers.event_parser import parse_events_html          # → GoalEvent, SubstitutionEvent

# Transformaciones — objetos → DataFrames Parquet
from transform.dim_equipos import update_dim_equipos
from transform.dim_partidos import update_dim_partidos
from transform.dim_jugadores import update_dim_jugadores
from transform.fact_goals import update_fact_goals
from transform.fact_substitutions import update_fact_substitutions
from transform.fact_appearances import update_fact_appearances

# Pipeline — orquestación
from pipeline.run_pipeline import run_incremental, run_full_pipeline
```

---
## 14. Características Técnicas

### Scraping robusto
- `requests.Session` con reintentos automáticos y backoff exponencial (urllib3 Retry)
- Delays aleatorios entre requests (1-3 s) para respetar los límites de BeSoccer
- Timeout configurable; errores HTTP capturados sin detener el pipeline

### Caché HTML permanente
- Todo HTML descargado se guarda en `html/lineups/` o `html/events/`
- Las ejecuciones semanales reutilizan el caché — un partido nunca se re-scrapea
- Permite re-parsear datos retroactivamente sin acceso a la web

### Procesamiento incremental
- `meta/processed_matches.parquet` registra qué partidos ya han sido procesados
- `run_incremental()` solo procesa los partidos finalizados nuevos
- El upsert en Parquet actualiza registros existentes sin duplicados

### Claves únicas en tablas de hechos
- `FACT_GOALS`: clave `match_id + goal_index` — permite dos goles del mismo jugador en el mismo minuto
- `FACT_SUBSTITUTIONS`: clave `match_id + minute + player_in_sk`
- `FACT_APPEARANCES`: clave `match_id + player_sk`

### Identificación de jugadores
- `player_id` extraído del número al final del slug de la URL del perfil BeSoccer
- Para jugadores sin URL: hash MD5 negativo sobre nombre+equipo normalizados (sin tildes)
- `player_sk` siempre tiene valor — garantiza integridad referencial en Power BI

### Calidad de datos
- `logger.warning` cuando `team_id = None` en clasificación, partidos o eventos
- `minute_in` de suplentes tomado de `FACT_SUBSTITUTIONS` como fuente única

---
## 15. Conclusiones

Este proyecto demuestra la construcción de un pipeline ETL completo para datos
deportivos en categorías semiprofesionales, donde no existe API oficial ni datos
estructurados de terceros para esta competición.

1. **Extracción** robusta con caché HTML permanente y gestión de errores
2. **Transformación** a modelo dimensional con claves únicas garantizadas
3. **Carga** incremental en Parquet para análisis eficiente en Power BI

El sistema está diseñado para mantenimiento semanal: cada lunes basta con
ejecutar `run_incremental()` para incorporar los partidos de la nueva jornada.

---
*Proyecto desarrollado como parte del Máster en Sports Data Campus*

---
## Anexo — Reprocesado Completo desde Cero

> **Solo ejecutar si se necesita reconstruir todas las tablas** (por ejemplo, tras
> cambios importantes en el pipeline). En el uso normal semanal esto **no es necesario**.

El proceso borra las tablas Parquet procesadas y vuelve a construirlas desde cero.
Los HTMLs en `html/` **no se borran** — se reutilizan para no re-descargar nada de BeSoccer.

In [ ]:
# ============================================================
# ANEXO: Reprocesado completo — solo ejecutar si es necesario
# Descomentar todo el bloque para activar.
# ============================================================

# tablas = [
#     config.DIM_EQUIPOS_PATH,
#     config.DIM_PARTIDOS_PATH,
#     config.DIM_JUGADORES_PATH,
#     config.FACT_GOALS_PATH,
#     config.FACT_SUBSTITUTIONS_PATH,
#     config.FACT_APPEARANCES_PATH,
#     config.PROCESSED_MATCHES_PATH,
# ]
# for p in tablas:
#     if p.exists():
#         p.unlink()
#         print(f"Eliminado: {p.name}")
# print("Tablas borradas. Los HTMLs en caché se conservan.")
# from pipeline.run_pipeline import run_full_pipeline
# stats = run_full_pipeline(force_reprocess=True)
# print(stats)